# 03 - Retrieval: Ingredient Overlap, BM25, Dense, and Hybrid

**Inputs:**
- `data/processed/cleaned_recipes.parquet` — output of notebooks 01 and 02 (must have `canonical_ingredients` column)
- `data/processed/alias_map.json` — Filipino ingredient alias map
- `ingredient_utils.py` — shared normalisation helpers

**Outputs:**
- Four retrieval functions: `overlap_retrieve()`,`bm25_retrieve()`, `dense_retrieve()`, `hybrid_retrieve()`
- Smoke-test results on 4 sample queries (English, Tagalog, Taglish, mixed)

Run cells in order. No GPU is required; internet is needed on the first run to download the embedding model.

## 0. Paths (works from `notebooks/` or the repo root)


In [ ]:
import sys, os
from pathlib import Path

# Find the repo root by walking up from wherever the kernel started,
# instead of assuming the working folder is named "notebooks".
start = Path.cwd().resolve()
ROOT = next((p for p in (start, *start.parents) if (p / "src" / "ingredient_utils.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Could not find src/ingredient_utils.py - open this notebook from inside NLP-Mini-Project.")
os.chdir(ROOT)                          # so "data/processed/..." paths work
sys.path.insert(0, str(ROOT / "src"))   # so "import ingredient_utils" works
print("Working directory:", Path.cwd())


## 1. Install dependencies (run once, then restart the kernel if needed)

FAISS is optional; the dense retriever falls back to NumPy if it is unavailable.


In [ ]:
%pip install pandas pyarrow rank_bm25 rapidfuzz sentence-transformers


## 2. Load data and alias map


In [ ]:
import json
import pandas as pd

try:
    from ingredient_utils import load_alias_map, build_lookup, normalize_text
except ModuleNotFoundError:
    import sys
    from pathlib import Path
    here = Path.cwd().resolve()
    src = next(p for p in (here, *here.parents) if (p / "src" / "ingredient_utils.py").exists()) / "src"
    sys.path.insert(0, str(src))
    from ingredient_utils import load_alias_map, build_lookup, normalize_text

from retrieval_metrics import ranking_metrics

# recipe
df = pd.read_parquet("data/processed/cleaned_recipes.parquet")
print(f"Loaded {len(df)} recipes")
print("Columns:", df.columns.tolist())

# alias map
alias_map = load_alias_map("data/processed/alias_map.json")
lookup, collisions = build_lookup(alias_map)
print(f"\nAlias map: {len(alias_map)} canonical ingredients, {len(lookup)} lookup entries")
if collisions:
    print("Collisions (review if many):", collisions[:5])

# verify canonical ingredients
assert "canonical_ingredients" in df.columns, (
    "Missing 'canonical_ingredients' — run notebook 02 first and re-save the parquet."
)
print("\nSample canonical ingredients:")
df[["recipe_name", "canonical_ingredients"]].sample(3, random_state=42)


## 3. Query normalisation helper


In [ ]:
import re

def parse_query(query: str, lookup: dict) -> list[str]:
    """Normalise a raw ingredient query string -> list of canonical ingredient names.

    Splits on commas, semicolons, newlines, and 'at'/'and' conjunctions,
    then runs each token through the alias map.
    Items that don't map to a canonical are kept as-is so they can still
    appear in BM25 token matching.
    """
    # split on common separators
    parts = re.split(r"[,;\n]|\bat\b|\band\b", query, flags=re.IGNORECASE)
    canonicals = []
    unmapped   = []
    for part in parts:
        part = part.strip()
        if not part:
            continue
        c, left = normalize_text(part, lookup)
        for x in c:
            if x not in canonicals:
                canonicals.append(x)
        unmapped.extend(left)
    return canonicals, unmapped


# quick sanity check
for q in [
    "toyo, bawang, sibuyas, manok",
    "pork, sitaw, tomatoes, fish sauce",
    "baboy at kamatis, patis, suka",
    "chicken, garlic, ginger, coconut milk, lemongrass",
]:
    c, u = parse_query(q, lookup)
    print(f"  Query : {q}")
    print(f"  Canon : {c}")
    if u:
        print(f"  Unmapped tokens: {u}")
    print()

## 4. Retriever A — Ingredient Overlap

**How it works:** counts how many of the query's canonical ingredients appear in a recipe's `canonical_ingredients` list, then ranks by that count (descending). Ties are broken by coverage — `matched / recipe_total` — so shorter recipes that fully match beat long ones with the same raw count.

No index to build; runs in O(n) over the recipe list.

In [ ]:
def overlap_retrieve(
    query: str,
    df: pd.DataFrame,
    lookup: dict,
    top_k: int = 10,
) -> pd.DataFrame:
    """Rank recipes by canonical ingredient overlap with the query.

    Returns a DataFrame with columns:
        rank, recipe_id, recipe_name, matched_count, recipe_total,
        coverage, matched_ingredients, query_canonicals
    """
    query_canon, _ = parse_query(query, lookup)
    query_set = set(query_canon)

    if not query_set:
        print("[overlap] Warning: query produced no canonical ingredients.")
        return pd.DataFrame()

    rows = []
    for _, recipe in df.iterrows():
        recipe_set = set(recipe["canonical_ingredients"])
        matched     = query_set & recipe_set
        matched_cnt = len(matched)
        recipe_total = max(1, len(recipe_set))
        coverage     = matched_cnt / recipe_total          # what fraction of the recipe you can fill
        rows.append({
            "recipe_id"          : recipe["recipe_id"],
            "recipe_name"        : recipe["recipe_name"],
            "matched_count"      : matched_cnt,
            "recipe_total"       : recipe_total,
            "coverage"           : round(coverage, 4),
            "matched_ingredients": sorted(matched),
            "query_canonicals"   : query_canon,
        })

    result = (
        pd.DataFrame(rows)
        .sort_values(["matched_count", "coverage"], ascending=False)
        .head(top_k)
        .reset_index(drop=True)
    )
    result.insert(0, "rank", result.index + 1)
    return result


print("overlap_retrieve() defined — ready for testing.")

## 5. Retriever B — BM25

**How it works:** builds a BM25 index where each recipe's *document* is its list of canonical ingredient names (one token per ingredient). At query time the query's canonical names are used as the query tokens. BM25 rewards rare ingredients more than common ones (e.g. `lemongrass` matters more than `garlic`), which is a useful complement to raw overlap.

The index is built once and reused for all queries.

In [ ]:
from rank_bm25 import BM25Okapi

def build_bm25_index(df: pd.DataFrame) -> BM25Okapi:
    """Build a BM25 index from canonical_ingredients.

    Each recipe is a 'document' whose tokens are its canonical ingredient names
    (multi-word names are kept as single tokens so 'coconut milk' stays together).
    """
    corpus = [
        # replace spaces within ingredient names with underscores so
        # 'coconut milk' becomes one BM25 token, not two
        [ing.replace(" ", "_") for ing in row]
        for row in df["canonical_ingredients"]
    ]
    return BM25Okapi(corpus)


bm25_index = build_bm25_index(df)
print(f"BM25 index built over {len(df)} recipes.")
print(f"Avg document length: {bm25_index.avgdl:.1f} tokens")

In [ ]:
import numpy as np

def bm25_retrieve(
    query: str,
    df: pd.DataFrame,
    bm25_index: BM25Okapi,
    lookup: dict,
    top_k: int = 10,
) -> pd.DataFrame:
    """Rank recipes with BM25 over canonical ingredient tokens.

    Returns a DataFrame with columns:
        rank, recipe_id, recipe_name, bm25_score,
        matched_ingredients, query_canonicals
    """
    query_canon, _ = parse_query(query, lookup)
    query_tokens   = [ing.replace(" ", "_") for ing in query_canon]

    if not query_tokens:
        print("[bm25] Warning: query produced no canonical ingredients.")
        return pd.DataFrame()

    scores = bm25_index.get_scores(query_tokens)   # shape: (n_recipes,)

    top_indices = np.argsort(scores)[::-1][:top_k]

    rows = []
    query_set = set(query_canon)
    for rank, idx in enumerate(top_indices, start=1):
        recipe = df.iloc[idx]
        matched = sorted(query_set & set(recipe["canonical_ingredients"]))
        rows.append({
            "rank"               : rank,
            "recipe_id"          : recipe["recipe_id"],
            "recipe_name"        : recipe["recipe_name"],
            "bm25_score"         : round(float(scores[idx]), 4),
            "matched_ingredients": matched,
            "query_canonicals"   : query_canon,
        })

    return pd.DataFrame(rows)


print("bm25_retrieve() defined — ready for testing.")

## 6. Retriever C — Dense (multilingual sentence embeddings)

**How it works:** each recipe is turned into a short text (`recipe name + canonical ingredients`) and embedded with a sentence-embedding model. The query is embedded the same way and recipes are ranked by cosine similarity (embeddings are L2-normalised, so inner product = cosine).

- Default model is **mE5** (`intfloat/multilingual-e5-base`). **BGE-M3** and the English-only **all-MiniLM-L6-v2** are registered too, so they can be swapped in with `DENSE_MODEL_KEY` for the multilingual vs English-only comparison.
- Embeddings are cached to `data/processed/` so the model only encodes the recipes once.
- `use_alias=True` rewrites the query with the alias map first (e.g. `bawang` -> `garlic`). `use_alias=False` sends the raw query, which is what the with/without alias map experiment needs.

The first run downloads the model from Hugging Face, so it needs internet once. FAISS is optional; without it, retrieval uses a NumPy dot product.

In [ ]:
import hashlib
import numpy as np
from sentence_transformers import SentenceTransformer

try:
    import faiss
    HAS_FAISS = True
except ImportError:
    HAS_FAISS = False
print("FAISS available:", HAS_FAISS)

DENSE_MODELS = {
    "mE5"   : {"name": "intfloat/multilingual-e5-base",
               "query_prefix": "query: ", "passage_prefix": "passage: "},
    "BGE-M3": {"name": "BAAI/bge-m3",
               "query_prefix": "", "passage_prefix": ""},
    "MiniLM": {"name": "sentence-transformers/all-MiniLM-L6-v2",
               "query_prefix": "", "passage_prefix": ""},
}
DENSE_MODEL_KEY = "mE5"


def build_doc_text(recipe) -> str:
    """Text that represents one recipe for the embedding model."""
    ings = ", ".join(recipe["canonical_ingredients"])
    return f"{recipe['recipe_name']}. Ingredients: {ings}"


def build_dense_index(df: pd.DataFrame, model_key: str = DENSE_MODEL_KEY,
                      cache_dir: str = "data/processed") -> dict:
    """Embed every recipe once and return everything needed for dense search."""
    cfg   = DENSE_MODELS[model_key]
    model = SentenceTransformer(cfg["name"])

    texts = [cfg["passage_prefix"] + build_doc_text(r) for _, r in df.iterrows()]

    # the hash in the file name means the cache refreshes if the recipes change
    digest     = hashlib.md5("\n".join(texts).encode("utf-8")).hexdigest()[:8]
    cache_path = Path(cache_dir) / f"embeddings_{model_key}_{digest}.npy"

    if cache_path.exists():
        emb = np.load(cache_path)
        print(f"[dense] Loaded cached embeddings: {cache_path.name} {emb.shape}")
    else:
        print(f"[dense] Encoding {len(texts)} recipes with {cfg['name']} ...")
        emb = model.encode(texts, batch_size=64, normalize_embeddings=True,
                           show_progress_bar=True)
        emb = np.asarray(emb, dtype="float32")
        np.save(cache_path, emb)
        print(f"[dense] Saved embeddings to {cache_path.name}")

    faiss_index = None
    if HAS_FAISS:
        faiss_index = faiss.IndexFlatIP(emb.shape[1])
        faiss_index.add(emb)

    return {"key": model_key, "model": model, "cfg": cfg,
            "embeddings": emb, "faiss": faiss_index}


dense_index = build_dense_index(df, DENSE_MODEL_KEY)
print(f"Dense index ready: {dense_index['embeddings'].shape[0]} recipes, "
      f"{dense_index['embeddings'].shape[1]} dimensions")

In [ ]:
def build_dense_query_text(query: str, lookup: dict, use_alias: bool = True) -> str:
    """Query string that gets embedded. Falls back to the raw query if nothing maps."""
    if use_alias:
        canon, _ = parse_query(query, lookup)
        if canon:
            return "Ingredients: " + ", ".join(canon)
    return "Ingredients: " + query


def dense_retrieve(
    query: str,
    df: pd.DataFrame,
    dense_index: dict,
    lookup: dict,
    top_k: int = 10,
    use_alias: bool = True,
) -> pd.DataFrame:
    """Rank recipes by embedding similarity to the query.

    Returns a DataFrame with columns:
        rank, recipe_id, recipe_name, dense_score,
        matched_ingredients, query_canonicals
    """
    query_canon, _ = parse_query(query, lookup)
    query_text = dense_index["cfg"]["query_prefix"] + build_dense_query_text(query, lookup, use_alias)

    q = dense_index["model"].encode([query_text], normalize_embeddings=True)
    q = np.asarray(q, dtype="float32")

    k = min(top_k, len(df))
    if dense_index["faiss"] is not None:
        scores, idxs = dense_index["faiss"].search(q, k)
        scores, idxs = scores[0], idxs[0]
    else:
        all_scores = dense_index["embeddings"] @ q[0]
        idxs   = np.argsort(all_scores)[::-1][:k]
        scores = all_scores[idxs]

    query_set = set(query_canon)
    rows = []
    for rank, (idx, score) in enumerate(zip(idxs, scores), start=1):
        recipe  = df.iloc[int(idx)]
        matched = sorted(query_set & set(recipe["canonical_ingredients"]))
        rows.append({
            "rank"               : rank,
            "recipe_id"          : recipe["recipe_id"],
            "recipe_name"        : recipe["recipe_name"],
            "dense_score"        : round(float(score), 4),
            "matched_ingredients": matched,
            "query_canonicals"   : query_canon,
        })

    return pd.DataFrame(rows)


print("dense_retrieve() defined — ready for testing.")

## 7. Retriever D — Hybrid (Reciprocal Rank Fusion)

**How it works:** runs several retrievers, takes each one's ranked list, and gives every recipe a fused score

`RRF(recipe) = sum over methods of 1 / (k + rank_in_method)`

with `k = 60` (the value from Cormack et al., 2009). Recipes that rank high in more than one list rise to the top. No training is needed and the raw scores (BM25, cosine) never have to be put on the same scale.

The default hybrid is **BM25 + dense**. Add `"overlap"` to `methods` to fuse all three. Each method is asked for its top `depth` recipes; a recipe missing from a method's list simply gets no points from that method.

In [ ]:
def rrf_fuse(rank_lists: dict, rrf_k: int = 60, weights: dict = None) -> dict:
    """Fuse ranked lists of recipe_ids.

    rank_lists: {"bm25": [id, id, ...], "dense": [id, id, ...]} best-first.
    Returns {recipe_id: fused_score}.
    """
    weights = weights or {}
    fused = {}
    for method, ids in rank_lists.items():
        w = weights.get(method, 1.0)
        for rank, rid in enumerate(ids, start=1):
            fused[rid] = fused.get(rid, 0.0) + w / (rrf_k + rank)
    return fused


def hybrid_retrieve(
    query: str,
    df: pd.DataFrame,
    bm25_index: BM25Okapi,
    dense_index: dict,
    lookup: dict,
    top_k: int = 10,
    methods: tuple = ("bm25", "dense"),
    rrf_k: int = 60,
    depth: int = 100,
    weights: dict = None,
    dense_use_alias: bool = True,
) -> pd.DataFrame:
    """Fuse overlap / BM25 / dense rankings with Reciprocal Rank Fusion.

    Returns a DataFrame with columns:
        rank, recipe_id, recipe_name, rrf_score, <method>_rank for each method,
        matched_ingredients, query_canonicals
    """
    query_canon, _ = parse_query(query, lookup)
    query_set = set(query_canon)

    per_method = {}
    if "overlap" in methods:
        per_method["overlap"] = overlap_retrieve(query, df, lookup, top_k=depth)
    if "bm25" in methods:
        per_method["bm25"] = bm25_retrieve(query, df, bm25_index, lookup, top_k=depth)
    if "dense" in methods:
        per_method["dense"] = dense_retrieve(query, df, dense_index, lookup,
                                             top_k=depth, use_alias=dense_use_alias)

    # empty results (query had no canonical ingredients) just drop out of the fusion
    rank_lists = {m: list(r["recipe_id"]) for m, r in per_method.items() if not r.empty}
    if not rank_lists:
        print("[hybrid] Warning: no retriever returned results.")
        return pd.DataFrame()

    fused = rrf_fuse(rank_lists, rrf_k=rrf_k, weights=weights)
    ordered = sorted(fused.items(), key=lambda kv: kv[1], reverse=True)[:top_k]

    id_to_pos = {rid: i for i, rid in enumerate(df["recipe_id"])}
    method_ranks = {m: {rid: r for r, rid in enumerate(ids, start=1)} for m, ids in rank_lists.items()}

    rows = []
    for rank, (rid, score) in enumerate(ordered, start=1):
        recipe = df.iloc[id_to_pos[rid]]
        row = {
            "rank"       : rank,
            "recipe_id"  : rid,
            "recipe_name": recipe["recipe_name"],
            "rrf_score"  : round(score, 5),
        }
        for m in rank_lists:
            row[f"{m}_rank"] = method_ranks[m].get(rid)   # None if outside that method's top `depth`
        row["matched_ingredients"] = sorted(query_set & set(recipe["canonical_ingredients"]))
        row["query_canonicals"]    = query_canon
        rows.append(row)

    return pd.DataFrame(rows)


print("hybrid_retrieve() defined — ready for testing.")

## 8. Convenience wrapper ? run all four retrievers

Call `retrieve(query)` for overlap, BM25, dense, and hybrid results in one place.


In [ ]:
def retrieve(query: str, top_k: int = 10) -> dict:
    """Run all four retrievers and return their result DataFrames."""
    canon, unmapped = parse_query(query, lookup)
    print(f"Query : {query}")
    print(f"Canon : {canon}")
    if unmapped:
        print(f"Unmapped tokens (not in alias map): {unmapped}")
    print()

    results = {
        "overlap": overlap_retrieve(query, df, lookup, top_k=top_k),
        "bm25": bm25_retrieve(query, df, bm25_index, lookup, top_k=top_k),
        "dense": dense_retrieve(query, df, dense_index, lookup, top_k=top_k),
        "hybrid": hybrid_retrieve(query, df, bm25_index, dense_index, lookup, top_k=top_k),
    }
    columns = {
        "overlap": ["rank", "recipe_name", "matched_count", "recipe_total", "coverage"],
        "bm25": ["rank", "recipe_name", "bm25_score"],
        "dense": ["rank", "recipe_name", "dense_score"],
        "hybrid": ["rank", "recipe_name", "rrf_score", "bm25_rank", "dense_rank"],
    }
    for name, result in results.items():
        print(f"--- {name.title()} ---")
        if result.empty:
            print("No results.")
        else:
            print(result[columns[name]].to_string(index=False))
        print()
    return results


## 9. Notes

| | Ingredient Overlap | BM25 | Dense | Hybrid (RRF) |
|---|---|---|---|---|
| **Scoring** | count + coverage | IDF-weighted term frequency | cosine similarity of embeddings | sum of 1 / (60 + rank) over methods |
| **Needs alias map** | yes | yes | optional (`use_alias`) | yes (through BM25) |
| **Handles unseen wording** | no | no | yes | yes |
| **Index needed?** | no | yes | yes; embeddings are cached | yes; BM25 + dense |
| **Extra dependency** | none | `rank_bm25` | `sentence-transformers`; FAISS optional | BM25 + dense dependencies |

### RAG pipeline and evaluation
- Each retriever returns ranked `recipe_id`s. Notebook 04 uses those IDs to retrieve full recipe rows for the RAG context.
- For Recall@k, MRR, and nDCG@k, mask part of a known recipe's ingredients and check whether that recipe appears in the retrieved ranking.
- All retrievers treat ingredient quantities and preparation style as out of scope. Extending `alias_map.json` can improve coverage of unmapped Tagalog/Taglish terms.
- Dense recipe text includes the recipe name, so dish-name matches can boost dense scores. Dense scores are similarities, not probabilities.
- RRF ignores score magnitudes; its defaults (`rrf_k=60`, `depth=100`) are not tuned.


## 10. Compare all four retrievers on the same queries

Each query is run through ingredient overlap, BM25, dense, and hybrid retrieval. The matrix shows the number of shared recipe IDs in each pair's top-k results.


In [ ]:
SAMPLE_QUERIES = {
    "English": "chicken, garlic, onion, ginger, lemongrass, tamarind",
    "Tagalog": "baboy, bawang, sibuyas, suka, toyo, dahon ng laurel",
    "Taglish": "pork ribs, bawang, sibuyas, kamatis, patis",
    "Mixed": "chicken, luya, tanglad, coconut milk, patis, sili",
}
TOP_K = 5


def compare_retrievers(query: str, top_k: int = TOP_K) -> dict:
    results = {
        "Overlap": overlap_retrieve(query, df, lookup, top_k=top_k),
        "BM25": bm25_retrieve(query, df, bm25_index, lookup, top_k=top_k),
        "Dense": dense_retrieve(query, df, dense_index, lookup, top_k=top_k),
        "Hybrid": hybrid_retrieve(query, df, bm25_index, dense_index, lookup, top_k=top_k),
    }
    display_columns = {
        "Overlap": ["rank", "recipe_name", "matched_count", "coverage", "matched_ingredients"],
        "BM25": ["rank", "recipe_name", "bm25_score", "matched_ingredients"],
        "Dense": ["rank", "recipe_name", "dense_score", "matched_ingredients"],
        "Hybrid": ["rank", "recipe_name", "rrf_score", "bm25_rank", "dense_rank", "matched_ingredients"],
    }

    for name, result in results.items():
        print(f"[{name} ? top {top_k}]")
        if result.empty:
            print("No results.")
        else:
            print(result[display_columns[name]].to_string(index=False))
        print()

    names = list(results)
    recipe_ids = {
        name: set(result["recipe_id"]) if not result.empty else set()
        for name, result in results.items()
    }
    shared = pd.DataFrame(
        [[len(recipe_ids[left] & recipe_ids[right]) for right in names] for left in names],
        index=names,
        columns=names,
    )
    print("Shared recipe IDs in top-k (diagonal = result count):")
    print(shared)
    return results


for label, query in SAMPLE_QUERIES.items():
    print("=" * 80)
    print(f"QUERY [{label}]: {query}")
    canon, unmapped = parse_query(query, lookup)
    print(f"Canonicals: {canon}")
    if unmapped:
        print(f"Unmapped tokens: {unmapped}")
    print()
    compare_retrievers(query)


## 11. Effect of the alias map on dense retrieval (raw query vs alias-normalised query)

A Tagalog query sent to the embedding model as-is, compared with the same query after the alias map rewrites it to canonical names. This is the quick version of the with/without alias map experiment.

In [ ]:
ALIAS_TEST_QUERY = "baboy, bawang, sibuyas, suka, toyo, dahon ng laurel"
print(f"Query: {ALIAS_TEST_QUERY}\n")

raw_res   = dense_retrieve(ALIAS_TEST_QUERY, df, dense_index, lookup, top_k=5, use_alias=False)
alias_res = dense_retrieve(ALIAS_TEST_QUERY, df, dense_index, lookup, top_k=5, use_alias=True)

print("Dense, raw query (no alias map):")
print(raw_res[["rank", "recipe_name", "dense_score"]].to_string(index=False))
print()
print("Dense, alias-normalised query:")
print(alias_res[["rank", "recipe_name", "dense_score"]].to_string(index=False))

## 12. Multilingual vs English-only embeddings (optional)

Builds a second dense index with `all-MiniLM-L6-v2` and compares it with the default multilingual model on a raw Tagalog query (no alias map), which is where an English-only model should struggle. To compare BGE-M3 instead, change `"MiniLM"` to `"BGE-M3"` (larger download, slower first run).

In [ ]:
# Set to True to download and compare an additional embedding model.
RUN_OPTIONAL_MODEL_COMPARISON = False

if RUN_OPTIONAL_MODEL_COMPARISON:
    ENGLISH_ONLY_KEY = "MiniLM"
    english_index = build_dense_index(df, ENGLISH_ONLY_KEY)
    model_query = "baboy, bawang, sibuyas, suka, toyo, dahon ng laurel"

    multilingual = dense_retrieve(model_query, df, dense_index, lookup, top_k=5, use_alias=False)
    english_only = dense_retrieve(model_query, df, english_index, lookup, top_k=5, use_alias=False)

    print(f"Query (raw, no alias map): {model_query}\n")
    print(f"{DENSE_MODEL_KEY} (multilingual):")
    print(multilingual[["rank", "recipe_name", "dense_score"]].to_string(index=False))
    print()
    print(f"{ENGLISH_ONLY_KEY} (English-only):")
    print(english_only[["rank", "recipe_name", "dense_score"]].to_string(index=False))


## 13. Offline retrieval metrics (masked-ingredient benchmark)

There is no labeled set of real user queries yet, so this reproducible proxy hides part of each recipe's ingredients and uses that recipe ID as the relevant result. The table reports Precision@k, Recall@k, MRR@k, nDCG@k, and mean latency for the same sampled queries and each of the four retrievers. Similar recipes are not labeled relevant, so treat these as comparative estimates; a labeled query set is needed for a definitive winner.


In [ ]:
import random
import time


def evaluate_retrievers(
    recipes: pd.DataFrame,
    max_queries: int = 100,
    query_fraction: float = 0.6,
    top_k: int = 5,
    seed: int = 42,
) -> pd.DataFrame:
    """Evaluate retrievers against recipes with a reproducible masked query."""
    if max_queries < 1:
        raise ValueError("max_queries must be at least 1")
    if top_k < 1:
        raise ValueError("top_k must be at least 1")
    if not 0 < query_fraction < 1:
        raise ValueError("query_fraction must be between 0 and 1")

    eligible = []
    for _, recipe in recipes.iterrows():
        ingredients = list(dict.fromkeys(recipe["canonical_ingredients"]))
        if len(ingredients) >= 2:
            eligible.append((recipe["recipe_id"], ingredients))
    if not eligible:
        raise ValueError("No recipes with at least two canonical ingredients to evaluate")

    rng = random.Random(seed)
    selected = rng.sample(eligible, min(max_queries, len(eligible)))
    retrievers = {
        "Overlap": lambda query: overlap_retrieve(query, recipes, lookup, top_k=top_k),
        "BM25": lambda query: bm25_retrieve(query, recipes, bm25_index, lookup, top_k=top_k),
        "Dense": lambda query: dense_retrieve(query, recipes, dense_index, lookup, top_k=top_k),
        "Hybrid": lambda query: hybrid_retrieve(
            query, recipes, bm25_index, dense_index, lookup, top_k=top_k
        ),
    }

    warmup_query = ", ".join(selected[0][1][:2])
    for retrieve_fn in retrievers.values():
        retrieve_fn(warmup_query)

    rows = []
    for target_id, ingredients in selected:
        visible_count = min(
            len(ingredients) - 1,
            max(1, round(len(ingredients) * query_fraction)),
        )
        query = ", ".join(rng.sample(ingredients, visible_count))

        for name, retrieve_fn in retrievers.items():
            started = time.perf_counter()
            result = retrieve_fn(query)
            latency_ms = (time.perf_counter() - started) * 1000
            ranked_ids = result["recipe_id"].tolist() if not result.empty else []
            scores = ranking_metrics({target_id}, ranked_ids, k=top_k)
            rows.append({
                "retriever": name,
                "query_id": target_id,
                **scores,
                "latency_ms": latency_ms,
            })

    return (
        pd.DataFrame(rows)
        .groupby("retriever")
        .agg(
            queries=("query_id", "nunique"),
            precision_at_k=("precision_at_k", "mean"),
            recall_at_k=("recall_at_k", "mean"),
            mrr_at_k=("mrr_at_k", "mean"),
            ndcg_at_k=("ndcg_at_k", "mean"),
            mean_latency_ms=("latency_ms", "mean"),
        )
        .sort_values(["recall_at_k", "mrr_at_k", "ndcg_at_k"], ascending=False)
        .reset_index()
    )


retrieval_metrics = evaluate_retrievers(df)
display(retrieval_metrics.round(4))
